# 01 - Data Ingestion

**Objective:** Load the Breast Cancer Wisconsin Diagnostic (WDBC) dataset, clean it, encode the target, and save processed output.

**Dataset:** Breast Cancer Wisconsin Diagnostic (UCI)

**Steps:**
1. Load raw data from `data/raw/wdbc.data`
2. Encode target variable (M→1, B→0)
3. Check data quality (missing values, dtypes)
4. Save processed data


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")


Libraries imported successfully


### Dataset Attributes

The WDBC dataset contains 30 real-valued features computed from digitized images of fine needle aspirate (FNA) of breast masses. Each feature describes characteristics of the cell nuclei present in the image.

**Measurements (10):**

- **Radius:** Mean distance from center to points on the perimeter
- **Texture:** Standard deviation of gray-scale values
- **Perimeter:** Total perimeter of the nucleus
- **Area:** Total area of the nucleus
- **Smoothness:** Local variation in radius lengths
- **Compactness:** Perimeter² / Area − 1.0
- **Concavity:** Severity of concave portions of the contour
- **Concave Points:** Number of concave portions of the contour
- **Symmetry:** Symmetry of the nucleus
- **Fractal Dimension:** "Coastline approximation" − 1

**Statistics (3 per measurement):**

- **Mean:** Average value across all cells in the sample
- **SE:** Standard error across cells
- **Worst:** Mean of the three largest values (most abnormal)

**Target:**

- **Diagnosis:** `M` = Malignant, `B` = Benign — binary classification (what we predict)
- **ID:** Patient identifier — not useful for modeling


In [4]:
RAW_DIR = Path("../data/raw")

col_names = ["ID", "Diagnosis"]
attributes_cont = [
    "Radius", "Texture", "Perimeter", "Area", "Smoothness",
    "Compactness", "Concavity", "ConcavePoints", "Symmetry", "FractalDimension",
]
for feat in attributes_cont:
    for stat in ["Mean", "SE", "Worst"]:
        col_names.append(f"{stat}_{feat}")

df = pd.read_csv(RAW_DIR / "wdbc.data", header=None, names=col_names)

print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())
print("\nDiagnosis distribution:")
print(df["Diagnosis"].value_counts())


Dataset shape: (569, 32)

First few rows:
         ID Diagnosis  Mean_Radius  SE_Radius  Worst_Radius  Mean_Texture  \
0    842302         M        17.99      10.38        122.80        1001.0   
1    842517         M        20.57      17.77        132.90        1326.0   
2  84300903         M        19.69      21.25        130.00        1203.0   
3  84348301         M        11.42      20.38         77.58         386.1   
4  84358402         M        20.29      14.34        135.10        1297.0   

   SE_Texture  Worst_Texture  Mean_Perimeter  SE_Perimeter  ...  \
0     0.11840        0.27760          0.3001       0.14710  ...   
1     0.08474        0.07864          0.0869       0.07017  ...   
2     0.10960        0.15990          0.1974       0.12790  ...   
3     0.14250        0.28390          0.2414       0.10520  ...   
4     0.10030        0.13280          0.1980       0.10430  ...   

   Worst_Concavity  Mean_ConcavePoints  SE_ConcavePoints  Worst_ConcavePoints  \
0          

In [7]:
df.describe()

,ID,Mean_Radius,SE_Radius,Worst_Radius,Mean_Texture,SE_Texture,Worst_Texture,Mean_Perimeter,SE_Perimeter,Worst_Perimeter,...,Worst_Concavity,Mean_ConcavePoints,SE_ConcavePoints,Worst_ConcavePoints,Mean_Symmetry,SE_Symmetry,Worst_Symmetry,Mean_FractalDimension,SE_FractalDimension,Worst_FractalDimension
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,3.037183e+07,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,1.250206e+08,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,8.670000e+03,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,8.692180e+05,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,9.060240e+05,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,8.813129e+06,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,9.113205e+08,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [9]:
print(df[col_names].notna().sum())

ID                        569
Diagnosis                 569
Mean_Radius               569
SE_Radius                 569
Worst_Radius              569
Mean_Texture              569
SE_Texture                569
Worst_Texture             569
Mean_Perimeter            569
SE_Perimeter              569
Worst_Perimeter           569
Mean_Area                 569
SE_Area                   569
Worst_Area                569
Mean_Smoothness           569
SE_Smoothness             569
Worst_Smoothness          569
Mean_Compactness          569
SE_Compactness            569
Worst_Compactness         569
Mean_Concavity            569
SE_Concavity              569
Worst_Concavity           569
Mean_ConcavePoints        569
SE_ConcavePoints          569
Worst_ConcavePoints       569
Mean_Symmetry             569
SE_Symmetry               569
Worst_Symmetry            569
Mean_FractalDimension     569
SE_FractalDimension       569
Worst_FractalDimension    569
dtype: int64


In [4]:
df["Diagnosis"] = df["Diagnosis"].apply(lambda x: 1 if x == "M" else 0)
df = df.drop(columns=["ID"])

print("\nTarget distribution:")
print(df["Diagnosis"].value_counts())
print(f"\nMalignant (1): {(df['Diagnosis']==1).sum()} ({(df['Diagnosis']==1).mean()*100:.1f}%)")
print(f"Benign (0): {(df['Diagnosis']==0).sum()} ({(df['Diagnosis']==0).mean()*100:.1f}%)")



Target distribution:
Diagnosis
0    357
1    212
Name: count, dtype: int64

Malignant (1): 212 (37.3%)
Benign (0): 357 (62.7%)


In [5]:
print(f"Missing values: {df.isnull().sum().sum()}")
print("\nData types:")
print(df.dtypes)


Missing values: 0

Data types:
Diagnosis                   int64
Mean_Radius               float64
SE_Radius                 float64
Worst_Radius              float64
Mean_Texture              float64
SE_Texture                float64
Worst_Texture             float64
Mean_Perimeter            float64
SE_Perimeter              float64
Worst_Perimeter           float64
Mean_Area                 float64
SE_Area                   float64
Worst_Area                float64
Mean_Smoothness           float64
SE_Smoothness             float64
Worst_Smoothness          float64
Mean_Compactness          float64
SE_Compactness            float64
Worst_Compactness         float64
Mean_Concavity            float64
SE_Concavity              float64
Worst_Concavity           float64
Mean_ConcavePoints        float64
SE_ConcavePoints          float64
Worst_ConcavePoints       float64
Mean_Symmetry             float64
SE_Symmetry               float64
Worst_Symmetry            float64
Mean_FractalDimen

In [6]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "clean_data.csv", index=False)
print("Clean data saved successfully")


Clean data saved successfully


In [ ]:
df_check = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
print(f"Loaded data shape: {df_check.shap e}")
assert df_check.shape == df.shape, "Shape mismatch after save-load cycle"
print("Verification passed: data round-trips correctly")


Loaded data shape: (569, 31)
Verification passed: data round-trips correctly


## 2. Relational Data — Joining Tables

In real-world projects, data is rarely handed to you in a single tidy CSV. 
It is often spread across multiple tables in a database — a **fact** table 
and several **dimension** tables, linked by key columns.

To build your feature matrix you need to **join** these tables back together.

In this section you will work with a relational version of the WDBC data,
split into three CSV files that simulate a real pathology database:

| File | Content | Key column |
|------|---------|------------|
| `patient_diagnosis.csv` | Patient ID + diagnosis label | `sample_id` |
| `mean_measurements.csv` | Mean + SE cell measurements (20 cols) | `mean_id` |
| `worst_measurements.csv` | Worst cell measurements (10 cols) | `worst_id` |

Some IDs appear in only some tables — exactly like a real database where lab 
results are missing for certain samples or extra records exist. 
Your job is to re-assemble the full dataset.

In [8]:
RELATIONAL_DIR = Path("../data/raw/relational")

diagnosis = pd.read_csv(RELATIONAL_DIR / "patient_diagnosis.csv")
mean_tbl = pd.read_csv(RELATIONAL_DIR / "mean_measurements.csv")
worst_tbl = pd.read_csv(RELATIONAL_DIR / "worst_measurements.csv")

print(f"Diagnosis: {diagnosis.shape}")
print(f"Mean: {mean_tbl.shape}")
print(f"Worst: {worst_tbl.shape}")
print(diagnosis.head())
print(mean_tbl.columns.tolist()[:5])


Diagnosis: (569, 2)
Mean: (562, 21)
Worst: (555, 11)
   sample_id Diagnosis
0          1         M
1          2         M
2          3         M
3          4         M
4          5         M
['mean_id', 'Mean_Radius', 'Mean_Texture', 'Mean_Perimeter', 'Mean_Area']


In [9]:
# LEFT JOIN - keep all 569 patients
merged = pd.merge(diagnosis, mean_tbl, left_on="sample_id", right_on="mean_id", how="left")
print(f"After LEFT JOIN: {merged.shape}")
print(f"Rows with missing mean/SE: {merged.isna().any(axis=1).sum()}")

# INNER JOIN - only patients with matching mean measurements
inner = pd.merge(diagnosis, mean_tbl, left_on="sample_id", right_on="mean_id", how="inner")
print(f"\nAfter INNER JOIN: {inner.shape}")
print(f"Rows lost with INNER: {len(diagnosis) - len(inner)}")


After LEFT JOIN: (569, 23)
Rows with missing mean/SE: 15

After INNER JOIN: (554, 23)
Rows lost with INNER: 15


In [10]:
# Chain second LEFT JOIN for worst measurements
full = pd.merge(merged, worst_tbl, left_on="sample_id", right_on="worst_id", how="left")
print(f"Full merge shape: {full.shape}")
print(full.head())

missing_worst = full['Worst_Radius'].isna().sum()
print(f"\nRows with missing Worst measurements: {missing_worst}")


Full merge shape: (569, 34)
   sample_id Diagnosis  mean_id  Mean_Radius  Mean_Texture  Mean_Perimeter  \
0          1         M      1.0        17.99         10.38          122.80   
1          2         M      2.0        20.57         17.77          132.90   
2          3         M      3.0        19.69         21.25          130.00   
3          4         M      4.0        11.42         20.38           77.58   
4          5         M      5.0        20.29         14.34          135.10   

   Mean_Area  Mean_Smoothness  Mean_Compactness  Mean_Concavity  ...  \
0     1001.0          0.11840           0.27760          0.3001  ...   
1     1326.0          0.08474           0.07864          0.0869  ...   
2     1203.0          0.10960           0.15990          0.1974  ...   
3      386.1          0.14250           0.28390          0.2414  ...   
4     1297.0          0.10030           0.13280          0.1980  ...   

   Worst_Radius  Worst_Texture  Worst_Perimeter  Worst_Area  Worst_Smo

In [11]:
# Count rows with all 30 measurements present
complete = full.dropna()
print(f"Complete rows after all joins: {len(complete)}")
print(f"Original single-table rows: 569")
print(f"\nMissing diagnosis-mean link: 15")
print(f"Missing diagnosis-worst link: 20")
print(f"Rows missing both: {569 - 15 - 20 - len(complete)}")


Complete rows after all joins: 535
Original single-table rows: 569

Missing diagnosis-mean link: 15
Missing diagnosis-worst link: 20
Rows missing both: -1


### Exercises

1. **Explore encoding options**: The Diagnosis column uses M=1, B=0. What happens if you reverse the mapping (M=0, B=1)? Does it change the model? Why or why not?
2. **Experiment with missing value handling**: Intentionally set a few values in the first feature column to NaN with `df.iloc[:10, 0] = np.nan`, then try both `dropna()` and `fillna(df.mean())` and compare the resulting shapes.
3. **Save in Parquet format**: Use `df.to_parquet()` instead of CSV and compare file sizes. Parquet is often faster and more space-efficient for numeric data.
4. **Feature engineering idea**: Create a new feature `size_ratio = Radius_Mean / Area_Mean`. Does this ratio capture something different from either raw measurement?
5. **What would break?**: If the raw data file had an extra column inserted between ID and Diagnosis, which part of this notebook would fail first? What if the first row contained column names instead of data?
6. **Which JOIN?**: You used LEFT JOINs to assemble the relational tables. Try replacing them with INNER JOINs. How many rows remain after the second merge? Why?
7. **Fake data detective**: The mean and worst measurement tables contain artificially generated rows (IDs > 569). Can you identify them? What heuristic did you use?
